# Accident Detection Agent
## 4-Stage Deep Learning Model
- Stage 1: YOLOv8 Vehicle Detection
- Stage 2: SimpleTracker Multi-Object Tracking
- Stage 3: ResNet50 Feature Extraction
- Stage 4: LSTM Sequence Classification

# Imports

In [ ]:
import os
import gc
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from torchvision import transforms, models
from ultralytics import YOLO

print("\n" + "="*70)
print("SYSTEM CONFIGURATION")
print("="*70)

DEVICE_DETECTION = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_CLASSIFIER = torch.device('cpu')

print(f"✓ CUDA Available: {torch.cuda.is_available()}")
print(f"  - Detection Device: {DEVICE_DETECTION}")
print(f"  - Classifier Device: {DEVICE_CLASSIFIER}")
if torch.cuda.is_available():
    print(f"  - GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"    GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  - CUDA Version: {torch.version.cuda}")

# Input Path Setup

In [ ]:
# Configure input/output paths for local runs
INPUT_ROOT = os.getcwd()
VIDEOS_PATH = os.path.join(INPUT_ROOT, "videos")
OUTPUT_DIR = os.path.join(INPUT_ROOT, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Single video override (set this to your file path)
SINGLE_VIDEO_PATH = None  # e.g. "C:/data/videos/my_video.mp4"

# Optional metadata CSV for batch runs
TEST_METADATA_PATH = None  # e.g. "C:/data/test_metadata.csv"

test_metadata = None
if TEST_METADATA_PATH and os.path.exists(TEST_METADATA_PATH):
    test_metadata = pd.read_csv(TEST_METADATA_PATH)
    print(f"✓ Test metadata loaded: {test_metadata.shape[0]} rows")
else:
    print("✓ No test metadata provided; using single video or videos folder")

print(f"✓ Output dir: {OUTPUT_DIR}")

In [ ]:
print("\n" + "="*70)
print("INPUT INSPECTION")
print("="*70)

PATH_COLUMN = None
ID_COLUMN = None

if test_metadata is not None:
    print(f"\nTest Metadata:")
    print(f"  Shape: {test_metadata.shape}")
    print(f"  Columns: {test_metadata.columns.tolist()}")
    print(f"\nSample rows:")
    print(test_metadata.head(2))

    # Check if metadata has 'path' column with video paths
    if 'path' in test_metadata.columns:
        PATH_COLUMN = 'path'
        print(f"\n✓ Found path column: 'path'")
        print(f"  Sample path: {test_metadata['path'].iloc[0]}")
    elif 'video_path' in test_metadata.columns:
        PATH_COLUMN = 'video_path'
        print(f"\n✓ Found path column: 'video_path'")
    else:
        print(f"\n⚠️  No 'path' column found")
        print(f"  All columns: {test_metadata.columns.tolist()}")

    # Find ID column (in case we need it)
    possible_id_cols = ['video_id', 'id', 'Video_ID', 'ID', 'video_name', 'name']
    for col in possible_id_cols:
        if col in test_metadata.columns:
            ID_COLUMN = col
            break

    if not ID_COLUMN:
        ID_COLUMN = test_metadata.columns[0]
    print(f"\n✓ Using ID column: '{ID_COLUMN}'")
else:
    print("\nNo test metadata provided.")
    if SINGLE_VIDEO_PATH:
        print(f"✓ Single video path set: {SINGLE_VIDEO_PATH}")
    elif VIDEOS_PATH and os.path.exists(VIDEOS_PATH):
        mp4_files = [f for f in os.listdir(VIDEOS_PATH) if f.lower().endswith('.mp4')]
        print(f"✓ Videos directory: {VIDEOS_PATH}")
        print(f"  Video files: {len(mp4_files)}")
        if mp4_files:
            print(f"  Sample file: {mp4_files[0]}")
    else:
        print("⚠️  No videos found. Set SINGLE_VIDEO_PATH or DATASET_NAME.")

print("="*70)

# Load Models

In [ ]:
print("\nLoading models...")

try:
    model_yolo = YOLO('yolov8m.pt')
    model_yolo = model_yolo.to(DEVICE_DETECTION)
    print(f"✓ YOLOv8 Medium")
except:
    model_yolo = YOLO('yolov8n.pt')
    model_yolo = model_yolo.to(DEVICE_DETECTION)

model_resnet = models.resnet50(pretrained=True)
model_resnet = torch.nn.Sequential(*list(model_resnet.children())[:-1])
model_resnet = model_resnet.to(DEVICE_DETECTION)
model_resnet.eval()
print(f"✓ ResNet50")

preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# LSTM Classifier

In [ ]:
TYPE_LABELS = ['rear-end', 't-bone', 'single', 'head-on', 'sideswipe']

class AccidentLSTM(torch.nn.Module):
    def __init__(self, input_size=2048, hidden_size=256, num_layers=2, num_classes=5):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.fc = torch.nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])

classifier = AccidentLSTM(input_size=2048, hidden_size=256, num_layers=2, num_classes=5)
classifier = classifier.to(DEVICE_CLASSIFIER)
classifier.eval()
print("✓ LSTM Classifier")
print(f"✓ Type labels: {TYPE_LABELS}")

# Video Processing

In [ ]:
def _format_timestamp(seconds):
    if seconds is None or seconds < 0:
        return "00:00"
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes:02d}:{secs:02d}"

class SimpleTracker:
    def __init__(self, max_age=30, min_hits=3):
        self.max_age = max_age
        self.min_hits = min_hits
        self.tracks = {}
        self.next_id = 0
    
    def update(self, detections):
        if len(detections) == 0:
            for tid in list(self.tracks.keys()):
                self.tracks[tid]['age'] += 1
                if self.tracks[tid]['age'] > self.max_age:
                    del self.tracks[tid]
            return []
        
        matched_tracks = []
        for det in detections:
            best_match = None
            best_dist = 50
            for tid, track in self.tracks.items():
                if track['hits'] >= self.min_hits:
                    dx = det['centroid'][0] - track['centroid'][0]
                    dy = det['centroid'][1] - track['centroid'][1]
                    dist = (dx**2 + dy**2)**0.5
                    if dist < best_dist:
                        best_dist = dist
                        best_match = tid
            
            if best_match is not None:
                self.tracks[best_match]['hits'] += 1
                self.tracks[best_match]['age'] = 0
                self.tracks[best_match]['centroid'] = det['centroid']
                matched_tracks.append(best_match)
            else:
                self.tracks[self.next_id] = {'hits': 1, 'age': 0, 'centroid': det['centroid']}
                matched_tracks.append(self.next_id)
                self.next_id += 1
        
        for tid in list(self.tracks.keys()):
            if tid not in matched_tracks:
                self.tracks[tid]['age'] += 1
                if self.tracks[tid]['age'] > self.max_age:
                    del self.tracks[tid]
        
        confirmed = []
        for tid, track in self.tracks.items():
            if track['hits'] >= self.min_hits:
                confirmed.append({'track_id': tid, 'centroid': track['centroid']})
        return confirmed

class VideoProcessor:
    def __init__(self, yolo, resnet, lstm, dev_det, dev_clf):
        self.yolo = yolo
        self.resnet = resnet
        self.lstm = lstm
        self.device_detection = dev_det
        self.device_classifier = dev_clf
        self.tracker = SimpleTracker(max_age=30, min_hits=3)
        self.preprocess = preprocess
    
    def _default_prediction(self):
        return {
            'accident_type': 2,
            'accident_type_label': 'single',
            'accident_time': 0.5,
            'center_x': 0.5,
            'center_y': 0.5,
            'confidence': 0.1,
            'vehicles_involved': 1,
            'bbox': None,
            'timestamp': "00:00",
            'frame_path': None
        }
    
    def process_video(self, video_path, sample_rate=2, save_frame_dir=None):
        try:
            # Reset tracker per video to avoid cross-video state leakage
            self.tracker = SimpleTracker(max_age=30, min_hits=3)

            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                return self._default_prediction()
            
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            fps = float(cap.get(cv2.CAP_PROP_FPS))
            if fps <= 1e-3:
                fps = 30.0
            
            frame_count = 0
            features_list = []
            centroids = []
            last_h = None
            last_w = None
            
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                last_h, last_w = frame.shape[:2]
                
                if frame_count % sample_rate != 0:
                    frame_count += 1
                    continue
                
                with torch.no_grad():
                    results = self.yolo(frame, conf=0.5)
                    detections = []
                    
                    for result in results:
                        for detection in result.boxes:
                            x1, y1, x2, y2 = [float(v) for v in detection.xyxy[0].tolist()]
                            detections.append({
                                'bbox': (x1, y1, x2, y2),
                                'centroid': ((x1 + x2) / 2, (y1 + y2) / 2)
                            })
                    
                    tracked = self.tracker.update(detections)

                    # Fallback: if no confirmed tracks yet, still use first detection
                    if not tracked and detections:
                        tracked = [{'track_id': -1, 'centroid': detections[0]['centroid']}]
                    
                    for track in tracked:
                        centroid = track['centroid']
                        h, w = frame.shape[:2]
                        x, y = int(centroid[0]), int(centroid[1])
                        crop_size = 100
                        x1 = max(0, x - crop_size)
                        y1 = max(0, y - crop_size)
                        x2 = min(w, x + crop_size)
                        y2 = min(h, y + crop_size)
                        crop = frame[y1:y2, x1:x2]
                        if crop.size == 0:
                            continue
                        crop = cv2.resize(crop, (224, 224))
                        image_tensor = self.preprocess(crop).unsqueeze(0).to(self.device_detection)
                        with torch.no_grad():
                            features = self.resnet(image_tensor).cpu().numpy().flatten()
                            features_list.append(features)
                            centroids.append(centroid)
                
                if len(features_list) >= 16:
                    break
                frame_count += 1
            
            cap.release()
            
            if len(features_list) == 0:
                return self._default_prediction()
            
            if len(features_list) < 16:
                features_list = [features_list[0]] * (16 - len(features_list)) + features_list
            else:
                features_list = features_list[:16]
                centroids = centroids[:16]
            
            features_tensor = torch.FloatTensor(features_list).unsqueeze(0).to(self.device_classifier)
            with torch.no_grad():
                logits = self.lstm(features_tensor)
                accident_type = int(logits.argmax(dim=1).item())
                confidence = float(torch.softmax(logits, dim=1).max().item())

            if 0 <= accident_type < len(TYPE_LABELS):
                accident_type_label = TYPE_LABELS[accident_type]
            else:
                accident_type_label = 'single'
            
            if centroids and last_h and last_w:
                center_x = float(min(1.0, np.mean([c[0] for c in centroids]) / last_w))
                center_y = float(min(1.0, np.mean([c[1] for c in centroids]) / last_h))
            else:
                center_x = 0.5
                center_y = 0.5
            
            accident_time = float(min(1.0, (frame_count * sample_rate) / total_frames)) if total_frames > 0 else 0.5
            timestamp_seconds = (accident_time * total_frames) / fps if total_frames > 0 else 0.0
            timestamp = _format_timestamp(timestamp_seconds)

            pred_center_px = None
            if last_h and last_w:
                pred_center_px = (center_x * last_w, center_y * last_h)
            
            # Run detection at the estimated accident timestamp for bbox and vehicle count
            vehicles_involved = 1
            incident_bbox = None
            incident_frame = None
            if total_frames > 0:
                target_frame = int(min(max(accident_time * total_frames, 0), max(total_frames - 1, 0)))
                cap2 = cv2.VideoCapture(video_path)
                if cap2.isOpened():
                    cap2.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
                    ret2, frame2 = cap2.read()
                    if ret2:
                        with torch.no_grad():
                            results = self.yolo(frame2, conf=0.5)
                            detections = []
                            best_area = 0.0
                            best_dist = None
                            for result in results:
                                for detection in result.boxes:
                                    x1, y1, x2, y2 = [float(v) for v in detection.xyxy[0].tolist()]
                                    area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
                                    detections.append((x1, y1, x2, y2, area))
                                    if pred_center_px is None:
                                        if area > best_area:
                                            best_area = area
                                            incident_bbox = (x1, y1, x2, y2)
                                    else:
                                        cx = (x1 + x2) / 2
                                        cy = (y1 + y2) / 2
                                        dist = ((cx - pred_center_px[0]) ** 2 + (cy - pred_center_px[1]) ** 2) ** 0.5
                                        if best_dist is None or dist < best_dist:
                                            best_dist = dist
                                            incident_bbox = (x1, y1, x2, y2)
                            vehicles_involved = int(max(1, len(detections)))
                            if incident_bbox and pred_center_px is not None and last_w and last_h:
                                diag = (last_w ** 2 + last_h ** 2) ** 0.5
                                max_dist = 0.35 * diag
                                if best_dist is not None and best_dist > max_dist:
                                    incident_bbox = None
                            if incident_bbox:
                                incident_frame = frame2.copy()
                    cap2.release()
            
            bbox = None
            if incident_bbox and last_w and last_h:
                x1, y1, x2, y2 = incident_bbox
                bbox = {
                    'x1': float(max(0.0, min(1.0, x1 / last_w))),
                    'y1': float(max(0.0, min(1.0, y1 / last_h))),
                    'x2': float(max(0.0, min(1.0, x2 / last_w))),
                    'y2': float(max(0.0, min(1.0, y2 / last_h)))
                }
            
            frame_path = None
            if save_frame_dir and incident_frame is not None:
                os.makedirs(save_frame_dir, exist_ok=True)
                base_name = os.path.splitext(os.path.basename(video_path))[0]
                frame_path = os.path.join(save_frame_dir, f"{base_name}_incident.jpg")
                cv2.imwrite(frame_path, incident_frame)
            
            return {
                'accident_type': accident_type,
                'accident_type_label': accident_type_label,
                'accident_time': accident_time,
                'center_x': center_x,
                'center_y': center_y,
                'confidence': confidence,
                'vehicles_involved': vehicles_involved,
                'bbox': bbox,
                'timestamp': timestamp,
                'frame_path': frame_path
            }
        except Exception:
            return self._default_prediction()

processor = VideoProcessor(model_yolo, model_resnet, classifier, DEVICE_DETECTION, DEVICE_CLASSIFIER)
print("✓ Processor initialized")

# Helper Functions

In [ ]:
def get_video_path(video_path_or_id):
    """Resolve video path robustly from path/id values in metadata."""
    if video_path_or_id is None:
        return None
    value = str(video_path_or_id).strip()

    # 1) Absolute path already exists
    if os.path.isabs(value) and os.path.exists(value):
        return value

    # 2) Relative path from metadata (e.g., videos/xxx.mp4)
    if INPUT_ROOT:
        candidate_relative = os.path.join(INPUT_ROOT, value)
        if os.path.exists(candidate_relative):
            return candidate_relative

    # 3) Treat value as ID and construct common layouts
    normalized_id = Path(value).stem
    possible_paths = []
    if VIDEOS_PATH:
        possible_paths.extend([
            os.path.join(VIDEOS_PATH, f"{normalized_id}.mp4"),
            os.path.join(VIDEOS_PATH, normalized_id, f"{normalized_id}.mp4"),
            os.path.join(VIDEOS_PATH, value),
        ])
    for path in possible_paths:
        if os.path.exists(path):
            return path

    return None

def collect_video_items():
    items = []
    if SINGLE_VIDEO_PATH:
        resolved = get_video_path(SINGLE_VIDEO_PATH)
        if resolved:
            items.append({"id": Path(resolved).stem, "path": resolved})
        return items

    if test_metadata is not None:
        for _, row in test_metadata.iterrows():
            if PATH_COLUMN:
                video_path_input = row[PATH_COLUMN]
            else:
                video_path_input = row[ID_COLUMN] if ID_COLUMN else None
            resolved = get_video_path(video_path_input)
            if resolved:
                items.append({"id": str(video_path_input), "path": resolved})
        return items

    if VIDEOS_PATH and os.path.exists(VIDEOS_PATH):
        for filename in sorted(os.listdir(VIDEOS_PATH)):
            if filename.lower().endswith('.mp4'):
                path = os.path.join(VIDEOS_PATH, filename)
                items.append({"id": Path(filename).stem, "path": path})
    return items

print(f"Videos path: {VIDEOS_PATH}")
if VIDEOS_PATH and os.path.exists(VIDEOS_PATH):
    print("✓ Found videos directory")
    mp4_count = len([f for f in os.listdir(VIDEOS_PATH) if f.lower().endswith('.mp4')])
    print(f"  Video files: {mp4_count}")
else:
    print("⚠️  Videos directory not found")

# Run Inference

In [ ]:
print("\n" + "="*70)
print("RUNNING INFERENCE")
print("="*70 + "\n")

predictions = []
videos_found = 0
videos_missing = 0

video_items = collect_video_items()
if not video_items:
    print("⚠️  No videos to process. Set SINGLE_VIDEO_PATH or provide videos folder.")

frame_dir = os.path.join(OUTPUT_DIR, "incident_frames")
os.makedirs(frame_dir, exist_ok=True)

for idx, item in enumerate(video_items, start=1):
    video_path = item["path"]
    if video_path is None or not os.path.exists(video_path):
        print(f"⚠️  Video not found: {item['id']}")
        videos_missing += 1
        pred = processor._default_prediction()
    else:
        try:
            if (idx + 1) % 10 == 0:
                print(f"Processing {idx}/{len(video_items)}: {item['id']}")
            pred = processor.process_video(video_path, sample_rate=2, save_frame_dir=frame_dir)
            videos_found += 1
        except Exception as e:
            print(f"❌ Error: {item['id']} - {str(e)[:50]}")
            pred = processor._default_prediction()
    
    pred['source_id'] = item["id"]
    pred['source_path'] = video_path
    predictions.append(pred)
    
    if idx % 10 == 0:
        gc.collect()

print(f"\n✓ Inference complete!")
print(f"  - Total videos: {len(predictions)}")
print(f"  - Videos found: {videos_found}")
print(f"  - Videos missing: {videos_missing}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
LLM_ENABLED = True
LLM_MAX_NEW_TOKENS = 96
LLM_TEMPERATURE = 0.4
LLM_TOP_P = 0.9
LLM_STYLE = "bullet_summary"  # bullet_summary or short_formal

llm_tokenizer = None
llm_model = None
hf_token = os.environ.get("HF_TOKEN")

def _pick_torch_dtype():
    if torch.cuda.is_available():
        if hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32

if LLM_ENABLED:
    try:
        llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, token=hf_token)
        if llm_tokenizer.pad_token is None:
            llm_tokenizer.pad_token = llm_tokenizer.eos_token
        llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME,
            token=hf_token,
            torch_dtype=_pick_torch_dtype(),
            device_map="auto"
        )
        llm_model.eval()
        print(f"✓ LLM loaded: {LLM_MODEL_NAME}")
    except Exception as e:
        print(f"⚠️  LLM not available: {str(e)[:120]}")
        llm_tokenizer = None
        llm_model = None

In [ ]:
import json

def _llm_generate(prompt_text):
    if llm_model is None or llm_tokenizer is None:
        return None
    inputs = llm_tokenizer(prompt_text, return_tensors="pt")
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=LLM_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=LLM_TEMPERATURE,
            top_p=LLM_TOP_P,
            pad_token_id=llm_tokenizer.eos_token_id
        )
    gen_tokens = output[0][input_len:]
    decoded = llm_tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return decoded if decoded else None

def _clean_report(text):
    if not text:
        return None
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    cleaned = []
    for line in lines:
        if line.lower().startswith("report:"):
            line = line.split(":", 1)[1].strip()
        cleaned.append(line)
    deduped = []
    for line in cleaned:
        if not deduped or line != deduped[-1]:
            deduped.append(line)
    return "\n".join(deduped).strip() if deduped else None


def _build_llm_prompt(event, severity, risk, notify):
    notify_text = ", ".join(notify) if notify else "none"
    fields = [
        f"Incident ID: {event['incident_id']}",
        f"Timestamp: {event['timestamp']}",
        f"Type: {event['accident_type']}",
        f"Vehicles involved: {event['vehicles_involved']}",
        f"Severity: {severity}",
        f"Injury risk: {risk['injury_risk']}",
        f"Road blocked: {'yes' if risk['road_blocked'] else 'no'}",
        f"Notify: {notify_text}",
        f"Confidence: {event.get('confidence', 0.0)}"
    ]
    user_msg = "Incident facts:\n- " + "\n- ".join(fields)
    if LLM_STYLE == "bullet_summary":
        system_msg = (
            "You are an emergency dispatcher. Output exactly 4 short bullets with key facts, "
            "then one short formal sentence summary. Max 90 words total. "
            "Do not invent injuries; if unknown say 'unknown'. No preamble."
        )
    else:
        system_msg = (
            "You are an emergency dispatcher. Write a short, formal incident report. "
            "Max 90 words. Do not invent injuries; if unknown say 'unknown'."
        )
    if hasattr(llm_tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ]
        return llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"{system_msg}\n\n{user_msg}\n\nReport:"

# Agent Step 1: Structured incident extraction
def build_incident_event(pred, idx):
    incident_id = f"INC_{idx:03d}"
    timestamp = pred.get('timestamp') or "00:00"
    accident_type = str(pred.get('accident_type_label', 'single')).strip().lower()
    vehicles_involved = int(pred.get('vehicles_involved', 1))
    confidence = float(pred.get('confidence', 0.0))
    event = {
        "incident_id": incident_id,
        "timestamp": timestamp,
        "accident_type": accident_type,
        "vehicles_involved": vehicles_involved,
        "confidence": round(confidence, 2)
    }
    if pred.get('bbox'):
        event["location_bbox"] = pred['bbox']
    if pred.get('frame_path'):
        event["frame_path"] = pred['frame_path']
    return event

# Agent Step 2: Severity estimation
def estimate_severity(event):
    base = {
        'rear-end': 1,
        'sideswipe': 1,
        'single': 1,
        't-bone': 2,
        'head-on': 3
    }
    score = base.get(event['accident_type'], 1)
    if event['vehicles_involved'] >= 3:
        score += 1
    if event['confidence'] >= 0.85:
        score += 1
    bbox = event.get('location_bbox')
    if bbox:
        area = max(0.0, (bbox['x2'] - bbox['x1']) * (bbox['y2'] - bbox['y1']))
        if area >= 0.2:
            score += 1
    if score <= 1:
        return "minor"
    if score == 2:
        return "moderate"
    if score == 3:
        return "severe"
    return "critical"

# Agent Step 3: Risk analysis
def analyze_risk(severity, event):
    if severity in ("severe", "critical"):
        injury_risk = "high"
    elif severity == "moderate":
        injury_risk = "medium"
    else:
        injury_risk = "low"
    road_blocked = severity in ("severe", "critical") or event['vehicles_involved'] >= 3
    return {
        "injury_risk": injury_risk,
        "road_blocked": bool(road_blocked)
    }

# Agent Step 4: Decision making
def decide_notifications(severity, risk, event):
    notify = []
    if severity == "minor":
        notify.append("traffic_police")
    else:
        notify.append("traffic_police")
        notify.append("ambulance")
    fire_risk = event['accident_type'] in ("head-on", "t-bone") and event['vehicles_involved'] >= 3
    if severity in ("severe", "critical") and (risk['road_blocked'] or fire_risk):
        notify.append("fire_department")
    return notify

# Agent Step 5: Explainable reasoning
def explain_reasoning(event, severity, risk, notify):
    reasons = []
    reasons.append(f"{severity} collision severity estimated")
    if event['accident_type'] in ("t-bone", "head-on"):
        reasons.append("high-impact collision type detected")
    if event['vehicles_involved'] >= 3:
        reasons.append("multiple vehicles involved")
    if risk['injury_risk'] in ("medium", "high"):
        reasons.append(f"injury risk rated {risk['injury_risk']}")
    if "fire_department" in notify:
        reasons.append("fire risk considered for high-impact multi-vehicle event")
    return reasons

# Agent Step 6: Natural-language report (LLM optional)
def generate_report(event, severity, risk, notify):
    notify_text = ", ".join(notify) if notify else "none"
    base_report = (
        f"A {severity} {event['accident_type']} collision involving {event['vehicles_involved']} vehicles "
        f"was detected at timestamp {event['timestamp']}. Injury risk is {risk['injury_risk']}; "
        f"road blockage: {'yes' if risk['road_blocked'] else 'no'}. Notified: {notify_text}."
    )
    prompt = _build_llm_prompt(event, severity, risk, notify)
    llm_text = _clean_report(_llm_generate(prompt))
    return llm_text if llm_text else base_report

# Agent Step 7: Build outputs and authority message
agentic_outputs = []
for idx, pred in enumerate(predictions, start=1):
    event = build_incident_event(pred, idx)
    severity = estimate_severity(event)
    risk = analyze_risk(severity, event)
    notify = decide_notifications(severity, risk, event)
    reasoning = explain_reasoning(event, severity, risk, notify)
    report = generate_report(event, severity, risk, notify)
    authority_message = f"Notify {', '.join(notify)}. {report}"
    payload = {
        "incident": event,
        "severity": {"severity": severity},
        "risk": risk,
        "decision": {"notify": notify},
        "reasoning": reasoning,
        "report": report,
        "authority_message": authority_message,
        "pipeline": {
            "accident_time": pred.get("accident_time"),
            "center_x": pred.get("center_x"),
            "center_y": pred.get("center_y"),
            "confidence": pred.get("confidence"),
            "bbox": pred.get("bbox"),
            "frame_path": pred.get("frame_path")
        },
    }
    agentic_outputs.append(payload)

agentic_path = os.path.join(OUTPUT_DIR, "agentic_incidents.jsonl")
with open(agentic_path, "w", encoding="utf-8") as f:
    for item in agentic_outputs:
        f.write(json.dumps(item) + "\n")

print(f"✓ Agentic outputs saved: {agentic_path}")
print("Sample agentic output:")
for item in agentic_outputs[:3]:
    print(json.dumps(item, indent=2))

In [ ]:
import matplotlib.pyplot as plt

def show_incident_preview(index=0):
    if not agentic_outputs:
        print("No agentic outputs available.")
        return
    item = agentic_outputs[index]
    print("Authority message:")
    print(item.get("authority_message", ""))
    print("\nReport:")
    print(item.get("report", ""))
    frame_path = item.get("incident", {}).get("frame_path")
    if frame_path and os.path.exists(frame_path):
        img = cv2.imread(frame_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.title(f"Incident Frame: {os.path.basename(frame_path)}")
        plt.axis("off")
        plt.show()
    else:
        print("No frame image available.")

show_incident_preview(0)

In [ ]:
print("="*70)
print("PIPELINE COMPLETE")
print("="*70)
print(f"Videos processed: {len(predictions)}")
print(f"Agentic outputs: {len(agentic_outputs)}")
print(f"Artifacts: {OUTPUT_DIR}")

PIPELINE COMPLETE
Videos: 2027
Submission: submission.csv
